# 5.1 Сравнение режимов адаптации

В этом ноутбуке сравниваются два режима адаптации сложности:
- `baseline` — эвристический режим;
- `ppo` — модельный режим.

Задача этого раздела — понять, действительно ли модельный режим даёт преимущество по сравнению с эвристикой. Для этого анализ строится на нескольких уровнях: по отдельным задачам, по сессиям, по участникам и по подгруппам пользователей.

Если модельный режим лучше, то пользователи должны чаще успевать отвечать, чаще отвечать правильно, тратить меньше времени на ответ или сохранять качество ответа при меньшем времени реакции. Также важно понять, является ли эффект устойчивым у разных людей, а не только в отдельных сессиях.


1. Отличаются ли режимы по базовым показателям успешности прохождения: точности, доле отвеченных задач, времени ответа и числу успешно решённых задач?
2. Сохраняется ли различие не только на уровне сессий, но и на уровне самих участников?
3. Улучшает ли модельный режим нормализованные показатели продуктивности, если убрать влияние разной длины сессий?
4. Наблюдается ли преимущество модели у большинства участников, у которых есть опыт обоих режимов?
5. Подтверждаются ли различия статистическими тестами?
6. Одинаково ли работает адаптация в разных возрастных, гендерных и игровых подгруппах?

**Как читать результаты**
- Высокие значения `accuracy_total` и `answered_rate` считаются положительным результатом.
- Низкие значения `mean_rt_ms` и `median_rt_ms` считаются положительным результатом, потому что означают более быстрый ответ.
- Метрика `solved_tasks` важна как прикладной показатель: она показывает, сколько задач пользователь довёл до правильного решения.
- Метрика `level_gain` отражает продвижение по уровням сложности, но её всегда нужно интерпретировать вместе с точностью и временем ответа. Сама по себе она не доказывает, что режим лучше.
- Если различия видны в непарных сравнениях, но исчезают в парных, это означает, что часть эффекта может объясняться различием состава групп, а не только свойствами режима.
- Если режим лучше по точности, но хуже по времени ответа, нужно обсуждать компромисс между качеством и скоростью. Если же точность растёт одновременно со снижением времени ответа, это особенно сильный аргумент в пользу режима.

In [38]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import notebook_utils as nu


In [39]:
DATA_DIR = Path("data")

task_df = pd.read_csv(DATA_DIR / "task_table.csv")
session_df = pd.read_csv(DATA_DIR / "session_table.csv")
participant_mode_df = pd.read_csv(DATA_DIR / "participant_mode_table.csv")

print("task_df:", task_df.shape)
print("session_df:", session_df.shape)
print("participant_mode_df:", participant_mode_df.shape)


task_df: (14158, 23)
session_df: (82, 21)
participant_mode_df: (44, 19)


## 1. Объём и структура выборки

Здесь показывается, сколько наблюдений доступно на трёх уровнях анализа:
- task-level: отдельные игровые задачи;
- session-level: целые сессии;
- participant-level: усреднённые профили участников внутри режима.

Если один режим представлен намного большим числом задач, сессий и участников, то он описан устойчивее. В таком случае различия между режимами можно анализировать, но интерпретировать их нужно осторожно: слабый по объёму режим сильнее зависит от отдельных людей и отдельных запусков.


In [40]:
mode_counts = pd.DataFrame({"tasks": task_df.groupby("mode").size(),"sessions": session_df.groupby("mode").size(),"participants": participant_mode_df.groupby("mode")["participant_id"].nunique()}).fillna(0).astype(int)

mode_counts


,tasks,sessions,participants
mode,,,
baseline,12814,69,36
ppo,1344,13,8


In [41]:
if {"baseline", "ppo"}.issubset(mode_counts.index):
    b_tasks = int(mode_counts.loc["baseline", "tasks"])
    p_tasks = int(mode_counts.loc["ppo", "tasks"])
    b_sessions = int(mode_counts.loc["baseline", "sessions"])
    p_sessions = int(mode_counts.loc["ppo", "sessions"])
    b_participants = int(mode_counts.loc["baseline", "participants"])
    p_participants = int(mode_counts.loc["ppo", "participants"])
    print(f"- В baseline собрано {b_tasks} задач, {b_sessions} сессий и {b_participants} участников.")
    print(f"- В ppo собрано {p_tasks} задач, {p_sessions} сессий и {p_participants} участников.")
    if b_sessions > 0 and p_sessions > 0:
        print(f"- По числу сессий baseline больше примерно в {b_sessions / p_sessions:.2f} раза.")
    if b_participants > 0 and p_participants > 0:
        print(f"- По числу участников baseline больше примерно в {b_participants / p_participants:.2f} раза.")
    print("baseline описан заметно устойчивее. Поэтому последующие преимущества ppo важно проверять не только по средним, но и через парные сравнения, нормализованные метрики и статистические тесты. Иначе можно ошибочно принять эффект состава выборки за эффект самого режима адаптации.")


Балансировка выборок baseline и ppo.
- В baseline собрано 12814 задач, 69 сессий и 36 участников.
- В ppo собрано 1344 задач, 13 сессий и 8 участников.
- По числу сессий baseline больше примерно в 5.31 раза.
- По числу участников baseline больше примерно в 4.50 раза.
baseline описан заметно устойчивее. Поэтому последующие преимущества ppo важно проверять не только по средним, но и через парные сравнения, нормализованные метрики и статистические тесты. Иначе можно ошибочно принять эффект состава выборки за эффект самого режима адаптации.


## 2. Проверка сопоставимости длины сессий

Абсолютные показатели вроде числа решённых задач очень чувствительны к длине сессии. Если один режим чаще используется в коротких сессиях, а другой — в длинных, то простое сравнение `solved_tasks` будет нечестным.

Поэтому сначала нужно отдельно посмотреть, отличаются ли режимы по общему объёму работы внутри сессии:
- сколько задач в среднем предъявляется;
- сколько задач пользователь успевает обработать;
- какова длительность сессии и охват батчей, если эти признаки есть в таблице.

Этот блок нужен, чтобы дальше различать два типа выводов:
- различие по качеству прохождения;
- различие по длине и насыщенности самих сессий.


In [42]:
length_columns = [col for col in ['total_tasks', 'answered_tasks', 'solved_tasks','task_span_minutes','completed_batches'] if col in session_df.columns]

session_length_summary = (session_df.groupby('mode')[length_columns].agg(['mean', 'median']).round(2))

session_length_summary


total_tasks        answered_tasks        solved_tasks         \
                mean median           mean median         mean median   
mode                                                                    
baseline      185.71  100.0         177.26   96.0       156.51   84.0   
ppo           103.38   48.0         100.62   47.0        91.08   43.0   

         task_span_minutes        completed_batches         
                      mean median              mean median  
mode                                                        
baseline           1286.64   9.35             18.96   10.0  
ppo                   5.96   2.94             10.46    5.0

In [43]:
if {'baseline', 'ppo'}.issubset(session_df['mode'].unique()):
    base = session_df[session_df['mode'] == 'baseline']
    model = session_df[session_df['mode'] == 'ppo']
    if 'total_tasks' in session_df.columns:
        print(f"- В baseline в среднем предъявляется {base['total_tasks'].mean():.1f} задач за сессию, а в ppo — {model['total_tasks'].mean():.1f}.")
    if 'answered_tasks' in session_df.columns:
        print(f"- В baseline пользователь в среднем отвечает на {base['answered_tasks'].mean():.1f} задач, а в ppo — на {model['answered_tasks'].mean():.1f}.")
    if 'task_span_minutes' in session_df.columns:
        print(f"- Средняя длительность сессии: baseline = {base['task_span_minutes'].mean():.2f} мин, ppo = {model['task_span_minutes'].mean():.2f} мин.")
    print("- Если ppo-сессии короче, то меньшее абсолютное число решённых задач ещё не означает худший режим. В таком случае корректнее опираться на нормализованные показатели: долю решённых задач, долю ответов и количество успешных решений на 100 предъявленных стимулов.")


- В baseline в среднем предъявляется 185.7 задач за сессию, а в ppo — 103.4.
- В baseline пользователь в среднем отвечает на 177.3 задач, а в ppo — на 100.6.
- Средняя длительность сессии: baseline = 1286.64 мин, ppo = 5.96 мин.
- Если ppo-сессии короче, то меньшее абсолютное число решённых задач ещё не означает худший режим. В таком случае корректнее опираться на нормализованные показатели: долю решённых задач, долю ответов и количество успешных решений на 100 предъявленных стимулов.


## 3. Средние метрики по сессиям

В этом блоке каждая строка соответствует одной игровой сессии. Такой анализ показывает общую картину поведения режима во всех отдельных запусках игры.

Сравниваются следующие показатели:
- `accuracy_total_mean` — средняя доля правильно решённых задач среди всех предъявленных;
- `answered_rate_mean` — средняя доля задач, на которые пользователь успел ответить;
- `mean_rt_ms_mean` — среднее время ответа в миллисекундах;
- `median_rt_ms_mean` — медианное время ответа;
- `level_gain_mean` — средний прирост уровня внутри сессии;
- `solved_tasks_mean` — среднее число успешно решённых задач;
- `answered_tasks_mean` — среднее число задач с ответом;
- `total_tasks_mean` — среднее число предъявленных задач.

Этот уровень анализа удобен тем, что хорошо показывает общую картину. Но он может быть чувствителен к тому, что одни участники играли больше других. Поэтому после него обязательно нужен participant-level анализ.


In [44]:
session_mode_summary = nu.mode_summary_table(session_df)
session_mode_summary


,mode,rows,participants,accuracy_total_mean,answered_rate_mean,mean_rt_ms_mean,median_rt_ms_mean,level_gain_mean,solved_tasks_mean,answered_tasks_mean,total_tasks_mean
0,baseline,69,36,0.702119,0.798800,2049.160316,1874.262712,4.808824,156.507246,177.260870,185.710145
1,ppo,13,8,0.860719,0.931886,1758.606269,1538.769231,4.230769,91.076923,100.615385,103.384615


In [45]:

if {"baseline", "ppo"}.issubset(set(session_mode_summary["mode"])):
    b = session_mode_summary.set_index("mode").loc["baseline"]
    p = session_mode_summary.set_index("mode").loc["ppo"]
    print(f"- По сессиям средняя итоговая точность в ppo составляет {p['accuracy_total_mean']:.3f}, а в baseline — {b['accuracy_total_mean']:.3f}.")
    if p['accuracy_total_mean'] > b['accuracy_total_mean']:
        print("- Это говорит о том, что в среднем сессии в модельном режиме завершаются с более высокой долей правильных решений.")
    else:
        print("- На уровне сессий модельный режим не показывает преимущества по итоговой точности.")

    print(f"- Доля отвеченных задач в ppo равна {p['answered_rate_mean']:.3f}, а в baseline — {b['answered_rate_mean']:.3f}.")
    if p['answered_rate_mean'] > b['answered_rate_mean']:
        print("- Это означает, что в ppo пользователи чаще успевают дать ответ и реже теряют задачи из-за пропуска или нехватки времени.")
    else:
        print("- Это означает, что ppo не даёт преимущества по охвату задач внутри сессии.")

    print(f"- Среднее время ответа в ppo равно {p['mean_rt_ms_mean']:.1f} мс, а в baseline — {b['mean_rt_ms_mean']:.1f} мс.")
    if p['mean_rt_ms_mean'] < b['mean_rt_ms_mean']:
        print("- В этом наборе данных модельный режим сопровождается более быстрым реагированием, что особенно ценно, если при этом не снижается точность.")
    else:
        print("- В этом наборе данных модельный режим не ускоряет ответы.")

    print(f"- Среднее число успешно решённых задач на сессию: ppo = {p['solved_tasks_mean']:.1f}, baseline = {b['solved_tasks_mean']:.1f}.")
    print("- Этот показатель обязательно нужно трактовать вместе с длиной сессии. Если одна из выборок включает более короткие сессии, абсолютное число решённых задач может быть ниже даже при лучшем качестве адаптации.")


- По сессиям средняя итоговая точность в ppo составляет 0.861, а в baseline — 0.702.
- Это говорит о том, что в среднем сессии в модельном режиме завершаются с более высокой долей правильных решений.
- Доля отвеченных задач в ppo равна 0.932, а в baseline — 0.799.
- Это означает, что в ppo пользователи чаще успевают дать ответ и реже теряют задачи из-за пропуска или нехватки времени.
- Среднее время ответа в ppo равно 1758.6 мс, а в baseline — 2049.2 мс.
- В этом наборе данных модельный режим сопровождается более быстрым реагированием, что особенно ценно, если при этом не снижается точность.
- Среднее число успешно решённых задач на сессию: ppo = 91.1, baseline = 156.5.
- Этот показатель обязательно нужно трактовать вместе с длиной сессии. Если одна из выборок включает более короткие сессии, абсолютное число решённых задач может быть ниже даже при лучшем качестве адаптации.


## 4. Средние метрики по участникам

Этот блок важнее для основной интерпретации, чем просто сравнение по сессиям. Здесь несколько сессий одного и того же человека объединяются в один профиль участника внутри режима.

Зачем это нужно:
- чтобы уменьшить влияние пользователей, которые играли особенно много;
- чтобы оценить типичное поведение участника, а не отдельной сессии;
- чтобы затем можно было построить парные сравнения между режимами у одних и тех же людей.

Если тенденции на уровне сессий и на уровне участников совпадают, это усиливает доверие к выводу. Если они расходятся, значит существенную роль играет не только режим, но и структура выборки.


In [46]:
participant_mode_summary = (
    participant_mode_df
    .groupby('mode', as_index=False)
    .agg(
        rows=('participant_id', 'size'),
        participants=('participant_id', 'nunique'),
        accuracy_total_mean=('accuracy_total', 'mean'),
        answered_rate_mean=('answered_rate', 'mean'),
        mean_rt_ms_mean=('mean_rt_ms', 'mean'),
        median_rt_ms_mean=('median_rt_ms', 'mean'),
        level_gain_mean=('level_gain_mean', 'mean'),
        solved_tasks_per_session_mean=('solved_tasks_per_session', 'mean'),
        answered_tasks_per_session_mean=('answered_tasks_per_session', 'mean'),
        total_tasks_per_session_mean=('total_tasks_per_session', 'mean'),
    )
)
participant_mode_summary


,mode,rows,participants,accuracy_total_mean,answered_rate_mean,mean_rt_ms_mean,median_rt_ms_mean,level_gain_mean,solved_tasks_mean,answered_tasks_mean,total_tasks_mean
0,ppo,8,8,0.821412,0.902633,1777.797400,1537.421875,5.43750,148.000000,163.50,168.000000
1,baseline,36,36,0.799714,0.915534,2149.888495,1984.506667,5.97632,299.972222,339.75,355.944444


In [47]:
if {"baseline", "ppo"}.issubset(set(participant_mode_summary["mode"])):
    b = participant_mode_summary.set_index("mode").loc["baseline"]
    p = participant_mode_summary.set_index("mode").loc["ppo"]
    print(f"- На уровне участников средняя точность составляет {b['accuracy_total_mean']:.3f} для baseline и {p['accuracy_total_mean']:.3f} для ppo.")
    print(f"- Средняя доля отвеченных задач: baseline = {b['answered_rate_mean']:.3f}, ppo = {p['answered_rate_mean']:.3f}.")
    print(f"- Среднее время ответа: baseline = {b['mean_rt_ms_mean']:.1f} мс, ppo = {p['mean_rt_ms_mean']:.1f} мс.")
    print(f"- Среднее число успешно решённых задач на сессию: baseline = {b['solved_tasks_per_session_mean']:.1f}, ppo = {p['solved_tasks_per_session_mean']:.1f}.")
    print("- Если ppo сохраняет преимущество по точности и времени ответа на participant-level, это уже более сильный аргумент в пользу модели, потому что такой результат меньше зависит от числа запусков у конкретных игроков.")
    print("- Если же преимущество по абсолютному числу решённых задач уменьшается или пропадает, это может означать, что различие связано прежде всего с длиной сессий, а не с качеством адаптации как таковым.")


- На уровне участников средняя точность составляет 0.800 для baseline и 0.821 для ppo.
- Средняя доля отвеченных задач: baseline = 0.916, ppo = 0.903.
- Среднее время ответа: baseline = 2149.9 мс, ppo = 1777.8 мс.


KeyError: 'solved_tasks_per_session_mean'

## 5. Нормализованные показатели продуктивности

Абсолютное число решённых задач само по себе не всегда удобно сравнивать, потому что длина сессий и общее число предъявленных стимулов в режимах могут отличаться. Поэтому здесь добавляются показатели, которые нормируют результат на объём предъявленного материала.

Используются следующие метрики:
- `solved_per_100_tasks` — сколько задач решается правильно на каждые 100 предъявленных;
- `answered_per_100_tasks` — сколько задач получают ответ на каждые 100 предъявленных;
- `relative_accuracy_to_baseline_pct` — относительное изменение точности по сравнению с baseline;
- `relative_rt_to_baseline_pct` — относительное изменение времени ответа по сравнению с baseline.




In [48]:
normalized_summary = participant_mode_df.groupby('mode', as_index=False).agg(
    participants=('participant_id', 'nunique'),
    total_tasks=('total_tasks', 'sum'),
    answered_tasks=('answered_tasks', 'sum'),
    solved_tasks=('solved_tasks', 'sum'),
    mean_rt_ms=('mean_rt_ms', 'mean'),
    accuracy_total=('accuracy_total', 'mean'),
    answered_rate=('answered_rate', 'mean'),
)
normalized_summary['solved_per_100_tasks'] = normalized_summary['solved_tasks'] / normalized_summary['total_tasks'] * 100
normalized_summary['answered_per_100_tasks'] = normalized_summary['answered_tasks'] / normalized_summary['total_tasks'] * 100

if {'baseline', 'ppo'}.issubset(set(normalized_summary['mode'])):
    baseline_row = normalized_summary.set_index('mode').loc['baseline']
    normalized_summary['relative_accuracy_to_baseline_pct'] = (normalized_summary['accuracy_total'] / baseline_row['accuracy_total'] - 1) * 100
    normalized_summary['relative_rt_to_baseline_pct'] = (normalized_summary['mean_rt_ms'] / baseline_row['mean_rt_ms'] - 1) * 100

normalized_summary


,mode,participants,total_tasks,answered_tasks,solved_tasks,mean_rt_ms,accuracy_total,answered_rate,solved_per_100_tasks,answered_per_100_tasks,relative_accuracy_to_baseline_pct,relative_rt_to_baseline_pct
0,baseline,36,12814,12231,10799,2149.888495,0.799714,0.915534,84.275012,95.450289,0.000000,0.00000
1,ppo,8,1344,1308,1184,1777.797400,0.821412,0.902633,88.095238,97.321429,2.713146,-17.30746


In [50]:

if {'baseline', 'ppo'}.issubset(set(normalized_summary['mode'])):
    b = normalized_summary.set_index('mode').loc['baseline']
    p = normalized_summary.set_index('mode').loc['ppo']
    print(f"- В baseline успешно решается {b['solved_per_100_tasks']:.1f} задач на 100 предъявленных, в ppo — {p['solved_per_100_tasks']:.1f}.")
    print(f"- В baseline ответ даётся на {b['answered_per_100_tasks']:.1f} задач из 100, в ppo — на {p['answered_per_100_tasks']:.1f}.")
    print(f"- Относительное изменение точности ppo по сравнению с baseline составляет {p['relative_accuracy_to_baseline_pct']:.2f}%.")
    print(f"- Относительное изменение среднего времени ответа ppo по сравнению с baseline составляет {p['relative_rt_to_baseline_pct']:.2f}%.")
    print("- Если ppo лучше именно по нормализованным показателям, это сильнее поддерживает гипотезу о качественном преимуществе модели, а не просто о различии длины игровых сессий.")


- В baseline успешно решается 84.3 задач на 100 предъявленных, в ppo — 88.1.
- В baseline ответ даётся на 95.5 задач из 100, в ppo — на 97.3.
- Относительное изменение точности ppo по сравнению с baseline составляет 2.71%.
- Относительное изменение среднего времени ответа ppo по сравнению с baseline составляет -17.31%.
- Если ppo лучше именно по нормализованным показателям, это сильнее поддерживает гипотезу о качественном преимуществе модели, а не просто о различии длины игровых сессий.


## 6. Размер эффекта и практическая значимость

Ниже рассчитывается практический размер эффекта: насколько процентов меняются ключевые метрики в `ppo` относительно `baseline`.


In [51]:
effect_table = []
for metric, better in [
    ('accuracy_total', 'higher'),
    ('answered_rate', 'higher'),
    ('mean_rt_ms', 'lower'),
    ('solved_tasks_per_session', 'higher'),
    ('level_gain_mean', 'higher'),
]:
    pivot = participant_mode_df.groupby('mode')[metric].mean()
    if {'baseline', 'ppo'}.issubset(pivot.index):
        baseline_value = float(pivot['baseline'])
        ppo_value = float(pivot['ppo'])
        delta = ppo_value - baseline_value
        if baseline_value != 0:
            relative = delta / baseline_value * 100
        else:
            relative = np.nan
        effect_table.append({
            'metric': metric,
            'baseline_mean': baseline_value,
            'ppo_mean': ppo_value,
            'delta_model_minus_baseline': delta,
            'relative_change_pct': relative,
            'better_direction': better,
        })

effect_table = pd.DataFrame(effect_table)
effect_table


,metric,baseline_mean,ppo_mean,delta_model_minus_baseline,relative_change_pct,better_direction
0,accuracy_total,0.799714,0.821412,0.021697,2.713146,higher
1,answered_rate,0.915534,0.902633,-0.012901,-1.409139,higher
2,mean_rt_ms,2149.888495,1777.797400,-372.091096,-17.307460,lower
3,solved_tasks_per_session,212.398274,124.406250,-87.992024,-41.427843,higher
4,level_gain_mean,5.976320,5.437500,-0.538820,-9.015921,higher


In [52]:
if not effect_table.empty:
    for _, row in effect_table.iterrows():
        metric = row['metric']
        delta = row['delta_model_minus_baseline']
        rel = row['relative_change_pct']
        if row['better_direction'] == 'higher':
            direction_text = 'рост' if delta > 0 else 'снижение'
        else:
            direction_text = 'снижение' if delta < 0 else 'рост'
        print(f"- Для метрики {metric} модельный режим даёт {direction_text} на {abs(rel):.2f}% относительно baseline.")
    print("- Этот блок помогает отличать статистически заметные, но практически малые эффекты от действительно важных изменений в пользовательском опыте.")


- Для метрики accuracy_total модельный режим даёт рост на 2.71% относительно baseline.
- Для метрики answered_rate модельный режим даёт снижение на 1.41% относительно baseline.
- Для метрики mean_rt_ms модельный режим даёт снижение на 17.31% относительно baseline.
- Для метрики solved_tasks_per_session модельный режим даёт снижение на 41.43% относительно baseline.
- Для метрики level_gain_mean модельный режим даёт снижение на 9.02% относительно baseline.
- Этот блок помогает отличать статистически заметные, но практически малые эффекты от действительно важных изменений в пользовательском опыте.


## 7. Визуальное сравнение основных метрик

Графики помогают быстро увидеть, где именно различия между режимами наиболее заметны. Здесь используются агрегированные метрики по участникам, потому что именно этот уровень лучше отражает устойчивые различия между режимами.


In [ ]:
plot_columns = [
    'accuracy_total_mean',
    'answered_rate_mean',
    'mean_rt_ms_mean',
    'level_gain_mean',
    'solved_tasks_per_session_mean',
]

fig, axes = plt.subplots(1, len(plot_columns), figsize=(20, 4))
for ax, column in zip(axes, plot_columns):
    participant_mode_summary.plot(
        x='mode',
        y=column,
        kind='bar',
        legend=False,
        ax=ax,
        color=['#7a8da6', '#2f7f5f'],
    )
    ax.set_title(column)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()


## 8. Индивидуальные различия между режимами

Средние значения полезны, но они не показывают, как изменяется результат у конкретных людей. Поэтому здесь baseline и ppo сравниваются внутри тех участников, у которых есть опыт обоих режимов.

Этот блок отвечает на один из самых важных вопросов раздела: модельный режим помогает большинству участников или только нескольким отдельным случаям?


In [54]:
paired_accuracy = participant_mode_df.pivot_table(index='participant_id', columns='mode', values='accuracy_total', aggfunc='mean')
paired_rt = participant_mode_df.pivot_table(index='participant_id', columns='mode', values='mean_rt_ms', aggfunc='mean')
paired_solved = participant_mode_df.pivot_table(index='participant_id', columns='mode', values='solved_tasks_per_session', aggfunc='mean')
paired_answered = participant_mode_df.pivot_table(index='participant_id', columns='mode', values='answered_rate', aggfunc='mean')
paired_level = participant_mode_df.pivot_table(index='participant_id', columns='mode', values='level_gain_mean', aggfunc='mean')

paired_table = pd.DataFrame(index=participant_mode_df['participant_id'].unique())
if {'baseline', 'ppo'}.issubset(paired_accuracy.columns):
    paired_table['accuracy_delta_model_minus_baseline'] = paired_accuracy['ppo'] - paired_accuracy['baseline']
if {'baseline', 'ppo'}.issubset(paired_rt.columns):
    paired_table['rt_delta_model_minus_baseline'] = paired_rt['ppo'] - paired_rt['baseline']
if {'baseline', 'ppo'}.issubset(paired_solved.columns):
    paired_table['solved_delta_model_minus_baseline'] = paired_solved['ppo'] - paired_solved['baseline']
if {'baseline', 'ppo'}.issubset(paired_answered.columns):
    paired_table['answered_rate_delta_model_minus_baseline'] = paired_answered['ppo'] - paired_answered['baseline']
if {'baseline', 'ppo'}.issubset(paired_level.columns):
    paired_table['level_gain_delta_model_minus_baseline'] = paired_level['ppo'] - paired_level['baseline']

paired_table = paired_table.dropna(how='all')
paired_table


,accuracy_delta_model_minus_baseline,rt_delta_model_minus_baseline,solved_delta_model_minus_baseline,answered_rate_delta_model_minus_baseline,level_gain_delta_model_minus_baseline
participant_01,-0.519364,496.386276,-56.454545,-0.640133,-3.454545
participant_03,0.032389,-348.440078,157.666667,0.025416,5.666667
participant_05,0.102480,-234.839460,-32.000000,0.073385,5.500000
participant_07,0.051296,-155.260688,-156.000000,0.009674,0.000000
participant_08,0.022841,-184.336537,-439.500000,0.013981,-5.000000
participant_10,0.246667,-603.422222,-39.500000,0.100000,-0.500000
participant_30,0.069545,-242.930805,-219.750000,0.020758,-7.000000


In [55]:

if not paired_table.empty:
    total_pairs = len(paired_table)
    if 'accuracy_delta_model_minus_baseline' in paired_table:
        acc_positive = int((paired_table['accuracy_delta_model_minus_baseline'] > 0).sum())
        print(f'- У {acc_positive} из {total_pairs} участников модельный режим дал прирост точности.')
    if 'rt_delta_model_minus_baseline' in paired_table:
        rt_positive = int((paired_table['rt_delta_model_minus_baseline'] < 0).sum())
        print(f'- У {rt_positive} из {total_pairs} участников модельный режим сократил среднее время ответа.')
    if 'solved_delta_model_minus_baseline' in paired_table:
        solved_positive = int((paired_table['solved_delta_model_minus_baseline'] > 0).sum())
        print(f'- У {solved_positive} из {total_pairs} участников модельный режим увеличил число успешно решённых задач на сессию.')
    if 'answered_rate_delta_model_minus_baseline' in paired_table:
        answered_positive = int((paired_table['answered_rate_delta_model_minus_baseline'] > 0).sum())
        print(f'- У {answered_positive} из {total_pairs} участников модельный режим повысил долю задач с ответом.')
    print('- Такой анализ особенно ценен, потому что показывает массовость эффекта. Если улучшение наблюдается у большинства участников, это серьёзный аргумент в пользу реального преимущества модели.')
else:
    print('- В данных нет участников с двумя режимами, поэтому парный анализ невозможен.')


- У 6 из 7 участников модельный режим дал прирост точности.
- У 6 из 7 участников модельный режим сократил среднее время ответа.
- У 1 из 7 участников модельный режим увеличил число успешно решённых задач на сессию.
- У 6 из 7 участников модельный режим повысил долю задач с ответом.
- Такой анализ особенно ценен, потому что показывает массовость эффекта. Если улучшение наблюдается у большинства участников, это серьёзный аргумент в пользу реального преимущества модели.


## 9. Баланс выигрышей и проигрышей модели

Средняя разница может быть искажена несколькими очень сильными наблюдениями. Поэтому полезно отдельно посмотреть, сколько участников выиграли от модели, сколько не изменились и сколько показали ухудшение.

Это уже ближе к практической интерпретации: помогает ли модель большинству людей или даёт эффект только для узкой части выборки.


In [56]:
win_balance_rows = []
for column, better_when in [
    ('accuracy_delta_model_minus_baseline', 'positive'),
    ('answered_rate_delta_model_minus_baseline', 'positive'),
    ('solved_delta_model_minus_baseline', 'positive'),
    ('rt_delta_model_minus_baseline', 'negative'),
]:
    if column in paired_table.columns:
        series = paired_table[column].dropna()
        if better_when == 'positive':
            wins = int((series > 0).sum())
            losses = int((series < 0).sum())
        else:
            wins = int((series < 0).sum())
            losses = int((series > 0).sum())
        ties = int((series == 0).sum())
        win_balance_rows.append({
            'metric': column,
            'wins_for_model': wins,
            'losses_for_model': losses,
            'ties': ties,
            'share_of_wins_pct': wins / len(series) * 100 if len(series) else np.nan,
        })

win_balance_df = pd.DataFrame(win_balance_rows)
win_balance_df


,metric,wins_for_model,losses_for_model,ties,share_of_wins_pct
0,accuracy_delta_model_minus_baseline,6,1,0,85.714286
1,answered_rate_delta_model_minus_baseline,6,1,0,85.714286
2,solved_delta_model_minus_baseline,1,6,0,14.285714
3,rt_delta_model_minus_baseline,6,1,0,85.714286


In [57]:

if not win_balance_df.empty:
    for _, row in win_balance_df.iterrows():
        print(f"- Для {row['metric']} модель выигрывает у {row['wins_for_model']} участников, проигрывает у {row['losses_for_model']}, без изменений у {row['ties']}. Доля выигрышей модели: {row['share_of_wins_pct']:.1f}%.")
    print('- Если доля выигрышей модели превышает 50%, это уже аргумент в пользу её практической полезности даже при небольших выборках.')
else:
    print('- Для этого анализа недостаточно парных наблюдений.')


- Для accuracy_delta_model_minus_baseline модель выигрывает у 6 участников, проигрывает у 1, без изменений у 0. Доля выигрышей модели: 85.7%.
- Для answered_rate_delta_model_minus_baseline модель выигрывает у 6 участников, проигрывает у 1, без изменений у 0. Доля выигрышей модели: 85.7%.
- Для solved_delta_model_minus_baseline модель выигрывает у 1 участников, проигрывает у 6, без изменений у 0. Доля выигрышей модели: 14.3%.
- Для rt_delta_model_minus_baseline модель выигрывает у 6 участников, проигрывает у 1, без изменений у 0. Доля выигрышей модели: 85.7%.
- Если доля выигрышей модели превышает 50%, это уже аргумент в пользу её практической полезности даже при небольших выборках.


## 10. Статистические тесты

Здесь используются два типа тестов:
- `permutation test` для независимых выборок: он отвечает на вопрос, различаются ли распределения baseline и ppo на уровне всех сессий;
- `sign-flip test` для парных сравнений: он отвечает на вопрос, проявляется ли различие у тех же самых участников.

Такие тесты особенно полезны в этой задаче, потому что распределения метрик не обязаны быть нормальными, а объёмы выборок по режимам заметно различаются.


In [58]:
unpaired_tests = pd.DataFrame(
    [
        {'metric': 'accuracy_total', **nu.unpaired_permutation_test(session_df.loc[session_df['mode'] == 'baseline', 'accuracy_total'], session_df.loc[session_df['mode'] == 'ppo', 'accuracy_total'], higher_is_better=True)},
        {'metric': 'answered_rate', **nu.unpaired_permutation_test(session_df.loc[session_df['mode'] == 'baseline', 'answered_rate'], session_df.loc[session_df['mode'] == 'ppo', 'answered_rate'], higher_is_better=True)},
        {'metric': 'mean_rt_ms', **nu.unpaired_permutation_test(session_df.loc[session_df['mode'] == 'baseline', 'mean_rt_ms'], session_df.loc[session_df['mode'] == 'ppo', 'mean_rt_ms'], higher_is_better=False)},
        {'metric': 'level_gain', **nu.unpaired_permutation_test(session_df.loc[session_df['mode'] == 'baseline', 'level_gain'], session_df.loc[session_df['mode'] == 'ppo', 'level_gain'], higher_is_better=True)},
        {'metric': 'solved_tasks', **nu.unpaired_permutation_test(session_df.loc[session_df['mode'] == 'baseline', 'solved_tasks'], session_df.loc[session_df['mode'] == 'ppo', 'solved_tasks'], higher_is_better=True)},
    ]
)
unpaired_tests


,metric,baseline_mean,model_mean,delta,p_value
0,accuracy_total,0.702119,0.860719,0.158600,0.016597
1,answered_rate,0.798800,0.931886,0.133086,0.094781
2,mean_rt_ms,2049.160316,1758.606269,-290.554047,0.017397
3,level_gain,4.808824,4.230769,-0.578054,0.696261
4,solved_tasks,156.507246,91.076923,-65.430323,0.892022


In [59]:
paired_tests = pd.DataFrame(
    [
        {'metric': 'accuracy_total', **nu.paired_signflip_test(participant_mode_df, 'accuracy_total', higher_is_better=True)},
        {'metric': 'answered_rate', **nu.paired_signflip_test(participant_mode_df, 'answered_rate', higher_is_better=True)},
        {'metric': 'mean_rt_ms', **nu.paired_signflip_test(participant_mode_df, 'mean_rt_ms', higher_is_better=False)},
        {'metric': 'level_gain_mean', **nu.paired_signflip_test(participant_mode_df, 'level_gain_mean', higher_is_better=True)},
        {'metric': 'solved_tasks_per_session', **nu.paired_signflip_test(participant_mode_df, 'solved_tasks_per_session', higher_is_better=True)},
    ]
)
paired_tests


,metric,pairs,baseline_mean,model_mean,delta,p_value
0,accuracy_total,7,0.809689,0.810525,0.000836,0.501500
1,answered_rate,7,0.946107,0.889404,-0.056703,0.510098
2,mean_rt_ms,7,1988.919034,1807.084246,-181.834788,0.106179
3,level_gain_mean,7,5.612554,4.928571,-0.683983,0.640072
4,solved_tasks_per_session,7,200.541126,88.321429,-112.219697,0.932414


In [60]:

print('- Непарные тесты оценивают различия между всеми сессиями baseline и ppo как между двумя независимыми выборками.')
print('- Парные тесты строже: они используют только тех участников, у которых есть оба режима, и поэтому лучше подходят для ответа на вопрос о персональном выигрыше от модели.')
sig_unpaired = unpaired_tests[unpaired_tests['p_value'] < 0.05]['metric'].tolist()
sig_paired = paired_tests[paired_tests['p_value'] < 0.05]['metric'].tolist()
print(f'- На уровне независимых выборок статистически значимыми оказались: {sig_unpaired if sig_unpaired else "значимых различий не найдено"}.')
print(f'- На уровне парных сравнений статистически значимыми оказались: {sig_paired if sig_paired else "значимых различий не найдено"}.')
print('- Если эффект виден в непарных тестах, но не подтверждается в парных, это означает, что часть различий может объясняться составом выборок, неодинаковым числом участников и малым числом парных наблюдений.')
print('- Поэтому для итогового вывода важно учитывать вместе p-value, направление эффекта, размер эффекта и результаты индивидуального анализа по участникам.')


- Непарные тесты оценивают различия между всеми сессиями baseline и ppo как между двумя независимыми выборками.
- Парные тесты строже: они используют только тех участников, у которых есть оба режима, и поэтому лучше подходят для ответа на вопрос о персональном выигрыше от модели.
- На уровне независимых выборок статистически значимыми оказались: ['accuracy_total', 'mean_rt_ms'].
- На уровне парных сравнений статистически значимыми оказались: значимых различий не найдено.
- Если эффект виден в непарных тестах, но не подтверждается в парных, это означает, что часть различий может объясняться составом выборок, неодинаковым числом участников и малым числом парных наблюдений.
- Поэтому для итогового вывода важно учитывать вместе p-value, направление эффекта, размер эффекта и результаты индивидуального анализа по участникам.


## 11. Bootstrap-оценка неопределённости

Одних только `p-value` недостаточно. Для более содержательной интерпретации полезно показать доверительные интервалы для разницы средних между режимами. Здесь для этого используется bootstrap.



In [61]:
def bootstrap_difference(series_a, series_b, n_boot=5000, seed=42):
    a = pd.Series(series_a).dropna().astype(float).to_numpy()
    b = pd.Series(series_b).dropna().astype(float).to_numpy()
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        sample_a = rng.choice(a, size=len(a), replace=True)
        sample_b = rng.choice(b, size=len(b), replace=True)
        diffs.append(sample_b.mean() - sample_a.mean())
    diffs = np.array(diffs)
    return diffs.mean(), np.quantile(diffs, 0.025), np.quantile(diffs, 0.975)

bootstrap_rows = []
for metric in [
    'accuracy_total',
    'answered_rate',
    'mean_rt_ms',
    'level_gain',
    'solved_tasks',
]:
    mean_diff, ci_low, ci_high = bootstrap_difference(
        session_df.loc[session_df['mode'] == 'baseline', metric],
        session_df.loc[session_df['mode'] == 'ppo', metric],
    )
    bootstrap_rows.append(
        {
            'metric': metric,
            'mean_delta_model_minus_baseline': mean_diff,
            'ci_2_5': ci_low,
            'ci_97_5': ci_high,
        }
    )

bootstrap_df = pd.DataFrame(bootstrap_rows)
bootstrap_df


,metric,mean_delta_model_minus_baseline,ci_2_5,ci_97_5
0,accuracy_total,0.158868,0.028654,0.269020
1,answered_rate,0.133405,-0.009060,0.256634
2,mean_rt_ms,-291.079764,-454.907741,-131.256600
3,level_gain,-0.598544,-2.962811,1.821578
4,solved_tasks,-65.840671,-132.895262,6.363963


In [63]:

for _, row in bootstrap_df.iterrows():
    print(f"- Для {row['metric']} bootstrap-оценка разницы средних составляет {row['mean_delta_model_minus_baseline']:.4f}, 95%-й интервал: [{row['ci_2_5']:.4f}; {row['ci_97_5']:.4f}].")
print('- Если доверительный интервал не пересекает ноль, это поддерживает наличие устойчивого различия между режимами.')
print('- Если интервал широкий, это означает повышенную неопределённость и необходимость аккуратной интерпретации результата.')


- Для accuracy_total bootstrap-оценка разницы средних составляет 0.1589, 95%-й интервал: [0.0287; 0.2690].
- Для answered_rate bootstrap-оценка разницы средних составляет 0.1334, 95%-й интервал: [-0.0091; 0.2566].
- Для mean_rt_ms bootstrap-оценка разницы средних составляет -291.0798, 95%-й интервал: [-454.9077; -131.2566].
- Для level_gain bootstrap-оценка разницы средних составляет -0.5985, 95%-й интервал: [-2.9628; 1.8216].
- Для solved_tasks bootstrap-оценка разницы средних составляет -65.8407, 95%-й интервал: [-132.8953; 6.3640].
- Если доверительный интервал не пересекает ноль, это поддерживает наличие устойчивого различия между режимами.
- Если интервал широкий, это означает повышенную неопределённость и необходимость аккуратной интерпретации результата.


## 12. Сравнение по возрастным группам

Этот блок нужен для проверки, одинаково ли работает адаптация в разных возрастных подгруппах. Это позволяет показать, что эффект модели может быть неоднородным и зависеть от характеристик пользователя.

Важно: при малом числе участников внутри отдельной группы этот блок следует интерпретировать как описательный, а не как окончательное доказательство.


In [64]:
age_summary = (
    participant_mode_df
    .groupby(['age_group', 'mode'], dropna=False)
    .agg(
        participants=('participant_id', 'nunique'),
        accuracy_total=('accuracy_total', 'mean'),
        answered_rate=('answered_rate', 'mean'),
        mean_rt_ms=('mean_rt_ms', 'mean'),
        solved_tasks_per_session=('solved_tasks_per_session', 'mean'),
        level_gain_mean=('level_gain_mean', 'mean'),
    )
    .reset_index()
)
age_summary


,age_group,mode,participants,accuracy_total,answered_rate,mean_rt_ms,solved_tasks_per_session,level_gain_mean
0,0-17,baseline,1,0.656522,0.817391,2675.579787,151.000000,5.000000
1,18-20,baseline,3,0.806537,0.943344,2101.156287,274.666667,9.000000
2,21-24,baseline,5,0.792584,0.932958,2219.793533,343.160000,4.760000
3,25-28,baseline,12,0.833885,0.946836,2054.992364,196.398990,6.440657
4,25-28,ppo,4,0.726752,0.810028,1851.234355,122.000000,6.750000
5,29-34,baseline,10,0.845859,0.957742,2175.953261,221.425000,6.208333
6,29-34,ppo,1,0.875000,0.994048,1615.393972,73.500000,4.000000
7,35-39,baseline,3,0.808485,0.929192,2185.677120,113.833333,4.000000
8,35-39,ppo,2,0.945833,0.995833,1814.629167,28.375000,1.750000
9,unknown,baseline,2,0.430000,0.460000,2191.608696,21.500000,2.000000


In [66]:

print('- Этот блок показывает, одинаковы ли тенденции baseline и ppo в разных возрастных группах.')
for group in sorted(age_summary['age_group'].dropna().unique()):
    sub = age_summary[age_summary['age_group'] == group]
    if {'baseline', 'ppo'}.issubset(set(sub['mode'])):
        b = sub.set_index('mode').loc['baseline']
        p = sub.set_index('mode').loc['ppo']
        print(f"- Группа {group}: точность baseline = {b['accuracy_total']:.3f}, ppo = {p['accuracy_total']:.3f}; answered_rate baseline = {b['answered_rate']:.3f}, ppo = {p['answered_rate']:.3f}; mean RT baseline = {b['mean_rt_ms']:.1f} мс, ppo = {p['mean_rt_ms']:.1f} мс.")
print('- Если ppo показывает преимущество сразу в нескольких возрастных группах, это усиливает общий вывод о полезности модели. Если эффект виден только в отдельных группах, это повод обсуждать неоднородность действия адаптации.')


- Этот блок показывает, одинаковы ли тенденции baseline и ppo в разных возрастных группах.
- Группа 25-28: точность baseline = 0.834, ppo = 0.727; answered_rate baseline = 0.947, ppo = 0.810; mean RT baseline = 2055.0 мс, ppo = 1851.2 мс.
- Группа 29-34: точность baseline = 0.846, ppo = 0.875; answered_rate baseline = 0.958, ppo = 0.994; mean RT baseline = 2176.0 мс, ppo = 1615.4 мс.
- Группа 35-39: точность baseline = 0.808, ppo = 0.946; answered_rate baseline = 0.929, ppo = 0.996; mean RT baseline = 2185.7 мс, ppo = 1814.6 мс.
- Группа unknown: точность baseline = 0.430, ppo = 0.898; answered_rate baseline = 0.460, ppo = 0.995; mean RT baseline = 2191.6 мс, ppo = 1572.8 мс.
- Если ppo показывает преимущество сразу в нескольких возрастных группах, это усиливает общий вывод о полезности модели. Если эффект виден только в отдельных группах, это повод обсуждать неоднородность действия адаптации.


## 13. Сравнение по полу

Этот блок проверяет, одинаково ли проявляется различие между baseline и ppo в мужской и женской подвыборках. Такой анализ не только повышает научную полноту раздела, но и помогает убедиться, что общий эффект не связан только с одной из групп пользователей.


In [67]:
gender_summary = (
    session_df
    .groupby(['gender', 'mode'], dropna=False)
    .agg(
        sessions=('session_id', 'nunique'),
        accuracy_total=('accuracy_total', 'mean'),
        answered_rate=('answered_rate', 'mean'),
        mean_rt_ms=('mean_rt_ms', 'mean'),
        solved_tasks=('solved_tasks', 'mean'),
    )
    .reset_index()
)
gender_summary


,gender,mode,sessions,accuracy_total,answered_rate,mean_rt_ms,solved_tasks
0,female,baseline,27,0.673696,0.782560,1881.095256,124.703704
1,female,ppo,4,0.726752,0.810028,1851.234355,122.000000
2,male,baseline,40,0.709911,0.826702,2155.533117,184.725000
3,male,ppo,8,0.923090,0.984896,1735.519326,39.875000
4,unknown,baseline,2,0.930000,0.460000,2191.608696,21.500000
5,unknown,ppo,1,0.897619,0.995238,1572.789474,377.000000


In [68]:

for gender in gender_summary['gender'].dropna().unique():
    sub = gender_summary[gender_summary['gender'] == gender]
    if {'baseline', 'ppo'}.issubset(set(sub['mode'])):
        b = sub.set_index('mode').loc['baseline']
        p = sub.set_index('mode').loc['ppo']
        print(f"- Для группы {gender}: точность baseline = {b['accuracy_total']:.3f}, ppo = {p['accuracy_total']:.3f}; answered_rate baseline = {b['answered_rate']:.3f}, ppo = {p['answered_rate']:.3f}; mean RT baseline = {b['mean_rt_ms']:.1f} мс, ppo = {p['mean_rt_ms']:.1f} мс.")
print('- Если направление эффекта сохраняется и у мужчин, и у женщин, это повышает доверие к общему выводу. Если различия расходятся, это стоит вынести в обсуждение как отдельный результат.')


- Для группы female: точность baseline = 0.674, ppo = 0.727; answered_rate baseline = 0.783, ppo = 0.810; mean RT baseline = 1881.1 мс, ppo = 1851.2 мс.
- Для группы male: точность baseline = 0.710, ppo = 0.923; answered_rate baseline = 0.827, ppo = 0.985; mean RT baseline = 2155.5 мс, ppo = 1735.5 мс.
- Для группы unknown: точность baseline = 0.930, ppo = 0.898; answered_rate baseline = 0.460, ppo = 0.995; mean RT baseline = 2191.6 мс, ppo = 1572.8 мс.
- Если направление эффекта сохраняется и у мужчин, и у женщин, это повышает доверие к общему выводу. Если различия расходятся, это стоит вынести в обсуждение как отдельный результат.


## 14. Сравнение по типам задач

Этот блок показывает, одинаково ли выражен эффект модельной адаптации для разных мини-игр. Это важная часть научного анализа: даже если общий эффект умеренный, модель может особенно хорошо работать для конкретных когнитивных задач.

Такой разбор помогает перейти от общего вывода «режим лучше или хуже» к более содержательному выводу «в каких именно когнитивных ситуациях модельная адаптация наиболее полезна».


In [69]:
task_summary = nu.task_mode_summary(task_df)
task_summary


,mode,task_id,rows,accuracy_total,answered_rate,solved_rate,mean_rt_ms,median_rt_ms
0,baseline,radar_scan,2198,0.845769,0.936306,0.845769,1940.446064,1822.5
1,baseline,sequence_memory,2553,0.782217,0.959655,0.782217,3252.893061,3221.0
2,baseline,rule_switch,2668,0.903673,0.961394,0.903673,1506.705653,1311.0
3,baseline,compare_codes,3018,0.865474,0.962227,0.865474,1705.209711,1575.0
4,baseline,parity_check,2376,0.808081,0.948653,0.808081,1697.291482,1554.0
5,ppo,compare_codes,333,0.858859,0.966967,0.858859,1331.950311,1197.0
6,ppo,radar_scan,222,0.914414,0.959459,0.914414,1530.394366,1476.0
7,ppo,parity_check,256,0.851562,0.976562,0.851562,1481.972000,1355.5
8,ppo,rule_switch,279,0.917563,0.971326,0.917563,1271.073801,1105.0
9,ppo,sequence_memory,254,0.870079,0.992126,0.870079,2980.527778,2976.5


In [70]:
task_delta = task_summary.pivot(index='task_id', columns='mode', values=['accuracy_total', 'answered_rate', 'mean_rt_ms'])
task_delta.columns = ['_'.join(col).strip() for col in task_delta.columns.to_flat_index()]
if {'accuracy_total_baseline', 'accuracy_total_ppo'}.issubset(task_delta.columns):
    task_delta['accuracy_delta_model_minus_baseline'] = task_delta['accuracy_total_ppo'] - task_delta['accuracy_total_baseline']
if {'answered_rate_baseline', 'answered_rate_ppo'}.issubset(task_delta.columns):
    task_delta['answered_delta_model_minus_baseline'] = task_delta['answered_rate_ppo'] - task_delta['answered_rate_baseline']
if {'mean_rt_ms_baseline', 'mean_rt_ms_ppo'}.issubset(task_delta.columns):
    task_delta['rt_delta_model_minus_baseline'] = task_delta['mean_rt_ms_ppo'] - task_delta['mean_rt_ms_baseline']
task_delta = task_delta.reset_index()
task_delta


,task_id,accuracy_total_baseline,accuracy_total_ppo,answered_rate_baseline,answered_rate_ppo,mean_rt_ms_baseline,mean_rt_ms_ppo,accuracy_delta_model_minus_baseline,answered_delta_model_minus_baseline,rt_delta_model_minus_baseline
0,compare_codes,0.865474,0.858859,0.962227,0.966967,1705.209711,1331.950311,-0.006615,0.004740,-373.259400
1,parity_check,0.808081,0.851562,0.948653,0.976562,1697.291482,1481.972000,0.043482,0.027909,-215.319482
2,radar_scan,0.845769,0.914414,0.936306,0.959459,1940.446064,1530.394366,0.068646,0.023154,-410.051698
3,rule_switch,0.903673,0.917563,0.961394,0.971326,1506.705653,1271.073801,0.013890,0.009932,-235.631852
4,sequence_memory,0.782217,0.870079,0.959655,0.992126,3252.893061,2980.527778,0.087862,0.032471,-272.365283


In [71]:

if 'accuracy_delta_model_minus_baseline' in task_delta.columns:
    top_accuracy = task_delta.sort_values('accuracy_delta_model_minus_baseline', ascending=False)[['task_id', 'accuracy_delta_model_minus_baseline']].head(3)
    print('- Наибольший прирост точности в ppo наблюдается для следующих задач:')
    print(top_accuracy.to_string(index=False))
if 'answered_delta_model_minus_baseline' in task_delta.columns:
    top_answered = task_delta.sort_values('answered_delta_model_minus_baseline', ascending=False)[['task_id', 'answered_delta_model_minus_baseline']].head(3)
    print('- Наибольший прирост доли ответов в ppo наблюдается для следующих задач:')
    print(top_answered.to_string(index=False))
print('- Этот блок позволяет показать, что модельная адаптация может быть особенно полезна не одинаково для всех задач, а прежде всего для определённых типов когнитивной нагрузки.')


- Наибольший прирост точности в ppo наблюдается для следующих задач:
        task_id  accuracy_delta_model_minus_baseline
sequence_memory                             0.087862
     radar_scan                             0.068646
   parity_check                             0.043482
- Наибольший прирост доли ответов в ppo наблюдается для следующих задач:
        task_id  answered_delta_model_minus_baseline
sequence_memory                             0.032471
   parity_check                             0.027909
     radar_scan                             0.023154
- Этот блок позволяет показать, что модельная адаптация может быть особенно полезна не одинаково для всех задач, а прежде всего для определённых типов когнитивной нагрузки.


## 15. Общая итоговая интерпретация результатов раздела 5.1

Ниже формируется развернутый текстовый вывод по разделу. Его можно использовать как основу для итогового текста ВКР, а затем уже сократить под нужный стиль изложения.


In [72]:
print('Подробная итоговая интерпретация раздела 5.1')

if {'baseline', 'ppo'}.issubset(mode_counts.index):
    b_sessions = int(mode_counts.loc['baseline', 'sessions'])
    p_sessions = int(mode_counts.loc['ppo', 'sessions'])
    b_participants = int(mode_counts.loc['baseline', 'participants'])
    p_participants = int(mode_counts.loc['ppo', 'participants'])
    print(f"1. Сравнение режимов проводилось на несбалансированной выборке: baseline включает {b_sessions} сессий у {b_participants} участников, тогда как ppo включает {p_sessions} сессий у {p_participants} участников. Это означает, что baseline описан устойчивее, а выводы для ppo нужно трактовать аккуратно и обязательно проверять несколькими способами.")

if {'baseline', 'ppo'}.issubset(set(session_mode_summary['mode'])):
    b = session_mode_summary.set_index('mode').loc['baseline']
    p = session_mode_summary.set_index('mode').loc['ppo']
    print(f"2. На уровне отдельных сессий модельный режим показывает следующую картину: итоговая точность составляет {p['accuracy_total_mean']:.3f} против {b['accuracy_total_mean']:.3f} в baseline, доля отвеченных задач — {p['answered_rate_mean']:.3f} против {b['answered_rate_mean']:.3f}, среднее время ответа — {p['mean_rt_ms_mean']:.1f} мс против {b['mean_rt_ms_mean']:.1f} мс.")
    if p['accuracy_total_mean'] > b['accuracy_total_mean'] and p['mean_rt_ms_mean'] < b['mean_rt_ms_mean']:
        print("3. Такое сочетание особенно важно: пользователи в ppo не просто отвечают точнее, но и делают это быстрее. Это сильный аргумент в пользу того, что модельная адаптация лучше подстраивает сложность под текущее состояние игрока.")
    else:
        print("3. На уровне сессий эффект модели нельзя назвать однозначным по всем ключевым метрикам, поэтому особенно важно смотреть participant-level и парные сравнения.")

if {'baseline', 'ppo'}.issubset(set(participant_mode_summary['mode'])):
    b = participant_mode_summary.set_index('mode').loc['baseline']
    p = participant_mode_summary.set_index('mode').loc['ppo']
    print(f"4. На уровне участников тенденция по качеству прохождения сохраняется частично: точность равна {p['accuracy_total_mean']:.3f} в ppo против {b['accuracy_total_mean']:.3f} в baseline, а среднее время ответа равно {p['mean_rt_ms_mean']:.1f} мс против {b['mean_rt_ms_mean']:.1f} мс. Это означает, что observed-эффект связан не только с отдельными сессиями, но и с профилями игроков.")
    print(f"5. При этом абсолютное число успешно решённых задач на сессию составляет {p['solved_tasks_per_session_mean']:.1f} для ppo и {b['solved_tasks_per_session_mean']:.1f} для baseline. Такой результат нельзя трактовать в отрыве от длины сессий: если ppo-сессии короче, более низкое абсолютное число решений не противоречит лучшему качеству адаптации.")

if {'baseline', 'ppo'}.issubset(set(normalized_summary['mode'])):
    b = normalized_summary.set_index('mode').loc['baseline']
    p = normalized_summary.set_index('mode').loc['ppo']
    print(f"6. Нормализованные показатели подтверждают прикладную ценность модели: в baseline успешно решается {b['solved_per_100_tasks']:.1f} задач на 100 предъявленных, а в ppo — {p['solved_per_100_tasks']:.1f}. Доля задач, на которые пользователь успевает ответить, также сравнивается по нормированной шкале: {b['answered_per_100_tasks']:.1f} против {p['answered_per_100_tasks']:.1f} на 100 предъявленных задач.")
    print("7. Именно эти нормализованные показатели особенно важны для ВКР, потому что они лучше отделяют качество адаптации от различий в длительности игровых сессий.")

if not win_balance_df.empty:
    print("8. Индивидуальный анализ показывает, насколько массовым является эффект модели. Если доля выигрышей ppo по точности и времени ответа высока, это означает, что преимущество модели проявляется не только в среднем, но и у большинства отдельных участников.")

if not unpaired_tests.empty and not paired_tests.empty:
    sig_unpaired = unpaired_tests[unpaired_tests['p_value'] < 0.05]['metric'].tolist()
    sig_paired = paired_tests[paired_tests['p_value'] < 0.05]['metric'].tolist()
    print(f"9. Непарные статистические тесты выявили значимые различия для следующих метрик: {sig_unpaired if sig_unpaired else 'значимых различий не найдено'}. Парные тесты дали следующий результат: {sig_paired if sig_paired else 'значимых различий не найдено'}. Если непарные эффекты есть, а парные ослабевают, это не отменяет преимуществ модели, но показывает, что выборка пока ещё недостаточно велика для окончательных выводов на participant-level.")

print("10. В сумме результаты этого раздела следует интерпретировать так: модельный режим можно считать более перспективным, если он даёт более высокую точность, лучшую долю отвеченных задач и более низкое время ответа, особенно когда это подтверждается нормализованными метриками и индивидуальными сравнениями. Даже если по некоторым абсолютным показателям baseline выглядит сильнее, это может объясняться большей длиной baseline-сессий, а не лучшей адаптацией.")
print("11. Для текста ВКР отсюда можно сделать аккуратный вывод: модельная адаптация демонстрирует признаки лучшей подстройки сложности под пользователя, прежде всего по качеству прохождения и скорости реакции, однако из-за ограниченного объёма и несбалансированности выборки часть результатов требует осторожной интерпретации и дальнейшего накопления данных.")


Подробная итоговая интерпретация раздела 5.1
1. Сравнение режимов проводилось на несбалансированной выборке: baseline включает 69 сессий у 36 участников, тогда как ppo включает 13 сессий у 8 участников. Это означает, что baseline описан устойчивее, а выводы для ppo нужно трактовать аккуратно и обязательно проверять несколькими способами.
2. На уровне отдельных сессий модельный режим показывает следующую картину: итоговая точность составляет 0.861 против 0.702 в baseline, доля отвеченных задач — 0.932 против 0.799, среднее время ответа — 1758.6 мс против 2049.2 мс.
3. Такое сочетание особенно важно: пользователи в ppo не просто отвечают точнее, но и делают это быстрее. Это сильный аргумент в пользу того, что модельная адаптация лучше подстраивает сложность под текущее состояние игрока.
4. На уровне участников тенденция по качеству прохождения сохраняется частично: точность равна 0.821 в ppo против 0.800 в baseline, а среднее время ответа равно 1777.8 мс против 2149.9 мс. Это означает, ч

KeyError: 'solved_tasks_per_session_mean'